# P21 EuroSAT: lane C (Person 3)

**Your job:** Scratch CNN (1%, 10%, 100%), ViT-Small at 100% labels (seed 0), then the scratch CNN pseudo-label students.
**Expected time:** about 3.5 h on a free Colab T4.

## How to run
1. Choose *Runtime → Change runtime type → T4 GPU*.
2. Choose *Runtime → Run all*. This notebook contains only your lane, so *Run all* is safe. It sets up, trains, and at the end gives you a zip file to download.
3. **Keep the browser tab open.** If Colab disconnects, reconnect and choose *Run all* again. Finished runs are saved on your Google Drive and are skipped, so nothing is lost.
4. When the last cell finishes, a file `lane_C.zip` downloads. **Send it to the project owner.** If the download does not start, the same files are in your Drive folder `p21_runs`; share that folder instead.

Do not edit any code or settings: every lane must use the same code version.
**Note:** The scratch CNN students need the scratch CNN teachers from the first command on this same machine.


In [ ]:
import os, sys, subprocess
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle/working")
BASE = "/content" if ON_COLAB else "/kaggle/working" if ON_KAGGLE else os.getcwd()
print(f"Colab: {ON_COLAB}  Kaggle: {ON_KAGGLE}  base dir: {BASE}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(out.stdout.strip() or "NO GPU FOUND")
assert out.returncode == 0 and out.stdout.strip(), (
    "No GPU. In Colab choose Runtime > Change runtime type > T4 GPU, then run this notebook again.")


## 1. Get the code and install

In [ ]:
REPO_URL = "https://github.com/ISHANT57/p21.git"   # public repo; for a zip upload instead, set this back to the <account>/<repo> placeholder
BRANCH = "main"
REPO = f"{BASE}/p21"

if not os.path.exists(REPO):
    if "<account>" not in REPO_URL:
        !git clone -b {BRANCH} {REPO_URL} {REPO}
    else:  # Option B: uploaded zip
        import glob
        zips = glob.glob(f"{BASE}/p21_code.zip") + glob.glob("/kaggle/input/*/p21_code.zip")
        assert zips, "Upload dist/p21_code.zip (see Option B) or set REPO_URL"
        !unzip -q -o {zips[0]} -d {BASE}
%cd {REPO}
!cat COMMIT 2>/dev/null || git log --oneline -1
!pip install -q -r requirements.txt -e .
sys.path.insert(0, f"{REPO}/src")   # make the editable install importable in this kernel


## 2. Keep results on Google Drive

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RUNS = "/content/drive/MyDrive/p21_runs"
    os.makedirs(DRIVE_RUNS, exist_ok=True)
    os.makedirs("results", exist_ok=True)
    if not os.path.islink("results/runs"):
        !rm -rf results/runs && ln -s {DRIVE_RUNS} results/runs
    !ls -la results/

## 3. Data: download, verify splits, pack the cache (about 15 min)

In [ ]:
!python scripts/download_data.py --which all --delete-zip
!python scripts/make_splits.py --check
!python scripts/build_cache.py --which all

## 4. Your runs (lane C)
The first cell lists what will run. The second cell runs it; it can take hours.

In [ ]:
!python scripts/run_grid.py --plan main --models scratch_cnn --fractions 0.01 0.1 --dry-run
!python scripts/run_grid.py --plan main --models scratch_cnn vit_small --fractions 1.0 --seeds 0 --dry-run
!python scripts/run_grid.py --plan pl --models scratch_cnn --fractions 0.01 0.1 --dry-run

In [ ]:
!python scripts/run_grid.py --plan main --models scratch_cnn --fractions 0.01 0.1 --max-hours 11
!python scripts/run_grid.py --plan main --models scratch_cnn vit_small --fractions 1.0 --seeds 0 --max-hours 11
!python scripts/run_grid.py --plan pl --models scratch_cnn --fractions 0.01 0.1 --max-hours 11

## 5. Check what finished

In [ ]:
!python scripts/run_grid.py --plan main --models scratch_cnn --fractions 0.01 0.1 --dry-run
!python scripts/run_grid.py --plan main --models scratch_cnn vit_small --fractions 1.0 --seeds 0 --dry-run
!python scripts/run_grid.py --plan pl --models scratch_cnn --fractions 0.01 0.1 --dry-run
!ls results/runs | wc -l

## 6. Zip the results and download
Send `lane_C.zip` to the project owner.

In [ ]:
!cd /content/drive/MyDrive && zip -qr /content/lane_C.zip p21_runs
from google.colab import files
files.download("/content/lane_C.zip")
